# Shadow removal figures -- SyMTRS

Shadowed `rgb_day` -> shadow-free `rgb_noshadow`, 3 000 test scenes at 2048 px. The rows
are whatever has been scored: the two GAN references (pix2pix, CycleGAN, trained at
256 px and resampled to 2048 before scoring) now, and ShadowFormer / Restormer / Uformer-B
/ ShadowMaskFormer as soon as their `per_image_test.json` exists -- re-run the notebook and
every figure picks them up, in the same colour slot they will always have.

The figures answer, in order: what does a removal look like (1, 2), where does the error
sit (3, 4), is the shadow lifted to the right brightness (5), how large is the gap
between shadow and lit pixels (6, 7), and does it depend on how much of the scene is in
shadow (8).

Nothing is re-scored here: the numbers are the evaluation's own (`per_image_test.json`,
written by `symtrs.model.shadow.runner.score_split`), the shadow mask is the one that
scoring used (`staging/test/mask`, derived from the rgb_day / rgb_noshadow pair), and the
colour space is `symtrs.metrics.shadow.rgb_to_lab`. Figures are written as PDF + PNG into
`notebooks/shadow_figures/`.

In [ ]:
from __future__ import annotations

import json
import sys
from functools import lru_cache
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon, Rectangle
from PIL import Image


def _repo_root() -> Path:
    """The repository, whether this runs as a notebook (no `__file__`) or as a script."""
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd().resolve()
    for anchor in (start, *start.parents):
        if (anchor / "symtrs" / "metrics" / "shadow.py").is_file():
            return anchor
    raise SystemExit(f"cannot find the SyMTRS repository above {start}")


REPO = _repo_root()
sys.path.insert(0, str(REPO))
from symtrs.metrics.shadow import rgb_to_lab  # noqa: E402 - the scorer's own colour space

RUNS = REPO / "runs" / "campaign2"
OUT = REPO / "notebooks" / "shadow_figures"
OUT.mkdir(exist_ok=True)
CACHE = Path("/tmp/shadowqual")  # derived, cheap to rebuild; never inside the dataset
CACHE.mkdir(exist_ok=True)

# ---------------------------------------------------------------------------
# Palette: the segmentation notebook's, validated there with the dataviz six-checks
# against white (6 categorical slots, adjacent pairs: all pass). Three slots sit below
# 3:1 on white, so every series is also directly labelled or has a legend entry.
# ---------------------------------------------------------------------------
SURFACE = "#ffffff"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
HAIR, AXIS = "#e1e0d9", "#c3c2b7"
BLUE_RAMP = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7",
             "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
SEQ = mcolors.LinearSegmentedColormap.from_list("symtrs_blue", BLUE_RAMP)
# Error maps start at white: a pixel the model got right should vanish into the page.
SEQ_ERR = mcolors.LinearSegmentedColormap.from_list("symtrs_err", ["#ffffff"] + BLUE_RAMP)
MASK_EDGE = "#eda100"  # the shadow outline, drawn on photographs; amber reads on grey and sky

#: (run directory, label). The order fixes the colour slot, so a model keeps its colour
#: whether or not the others have been scored yet.
METHODS = [
    ("shadowformer", "ShadowFormer"),
    ("restormer", "Restormer"),
    ("uformer", "Uformer-B"),
    ("shadowmaskformer", "ShadowMaskFormer"),
    ("pix2pix_shadow2noshadow", "pix2pix"),
    ("cyclegan_shadow2noshadow", "CycleGAN"),
]
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
COLOR = {run: c for (run, _), c in zip(METHODS, SLOTS)}
LABEL = dict(METHODS)
INPUT_C, TARGET_C = MUTED, INK

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 10.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelsize": 10, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": HAIR, "grid.linewidth": 0.7,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "legend.frameon": False, "legend.fontsize": 9,
    "lines.linewidth": 2.0, "lines.markersize": 6.5,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})


def tidy(ax, grid_axis="y"):
    for side in ("left", "bottom"):
        ax.spines[side].set_visible(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(axis=grid_axis)
    ax.grid(axis="x" if grid_axis == "y" else "y", visible=False)


def strip(ax, image, **kw):
    """An image pane with no ticks, no grid and no frame."""
    ax.imshow(image, interpolation="lanczos", **kw)
    ax.set_xticks([]), ax.set_yticks([])
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)


def save(fig, name):
    # pad_inches=0.01: the paper pays for every millimetre of margin.
    fig.savefig(OUT / f"{name}.pdf", bbox_inches="tight", pad_inches=0.01)
    fig.savefig(OUT / f"{name}.png", dpi=200, bbox_inches="tight", pad_inches=0.01)
    plt.close(fig)
    print(f"wrote shadow_figures/{name}.pdf / .png")
    try:  # inline when this runs as a notebook; no-op headless
        from IPython.display import Image as IPImage, display
        display(IPImage(filename=str(OUT / f"{name}.png"), width=760))
    except Exception:
        pass

In [ ]:
# ---------------------------------------------------------------------------
# Data: the scored runs, their per-image rows, and the shared staging
# ---------------------------------------------------------------------------
AVAILABLE = [run for run, _ in METHODS if (RUNS / run / "per_image_test.json").is_file()]
assert AVAILABLE, "no shadow run has been scored yet (no per_image_test.json under runs/campaign2)"
PER = {
    run: {row["key"]: row for row in json.loads((RUNS / run / "per_image_test.json").read_text())}
    for run in AVAILABLE
}
KEYS = sorted(set.intersection(*(set(rows) for rows in PER.values())))
# Every run stages the same split with the same mask settings; the first one's is used.
STAGING = RUNS / AVAILABLE[0] / "staging" / "test"
MASK_PARAMS = json.loads((STAGING / "mask_params.json").read_text())
for run in AVAILABLE[1:]:
    other = json.loads((RUNS / run / "staging" / "test" / "mask_params.json").read_text())
    assert (other["threshold"], other["open"]) == (MASK_PARAMS["threshold"], MASK_PARAMS["open"]), \
        f"{run} was scored with another shadow mask; its rows are not comparable"


def column(run, metric):
    return np.array([np.nan if PER[run][k][metric] is None else PER[run][k][metric] for k in KEYS])


FRACTION = column(AVAILABLE[0], "shadow_fraction")


def altitude_m(key):
    """Camera height from the key (`z+007837` is centimetres)."""
    return int(key.split("_z")[1]) / 100


print(f"scored: {', '.join(LABEL[r] for r in AVAILABLE)}  ({len(KEYS)} common test scenes)")
print(f"shadow mask: darkening > {MASK_PARAMS['threshold']}, opening {MASK_PARAMS['open']} px; "
      f"shadow fraction per scene: median {np.median(FRACTION):.1%}, "
      f"5-95 % {np.percentile(FRACTION, 5):.1%} - {np.percentile(FRACTION, 95):.1%}")
print(f"\n{'':18}{'PSNR':>7}{'shadow':>8}{'lit':>7}{'SSIM-Y':>8}{'MAE-LAB':>9}")
for run in AVAILABLE:
    print(f"{LABEL[run]:<18}" + "".join(
        f"{np.nanmean(column(run, m)):{w}.{p}f}" for m, w, p in
        (("psnr", 7, 2), ("psnr_shadow", 8, 2), ("psnr_nonshadow", 7, 2), ("ssim_y", 8, 3), ("mae_lab", 9, 2))))

In [ ]:
# ---------------------------------------------------------------------------
# Images. Everything is read at 2048 and reduced here, never re-generated.
#
# Prediction images are kept under `eval.retain_predictions`: `predictions/test/<key>.png`
# is a 512 px copy for most scenes, and the scenes the figures zoom into - one typical
# scene per camera height, chosen by `symtrs.model.shadow.runner.preview_keys` - are kept at
# full resolution in `preview/full/`. `load` prefers the full-resolution copy; a scene
# outside that set is still drawn, from its 512 px copy.
# ---------------------------------------------------------------------------
FULLRES = set.intersection(*[
    {p.name[len("test_"):-len(".png")] for p in (RUNS / run / "preview" / "full").glob("test_*.png")}
    for run in AVAILABLE
]) if AVAILABLE else set()


@lru_cache(maxsize=256)
def load(source, key, size=2048):
    """`source` is 'input', 'target', 'mask' or a run directory; float RGB in [0, 1]
    (bool for the mask), box-filtered to `size`."""
    if source in ("input", "target", "mask"):
        path = STAGING / source / f"{key}.png"
    else:
        full = RUNS / source / "preview" / "full" / f"test_{key}.png"
        path = full if full.is_file() else RUNS / source / "predictions" / "test" / f"{key}.png"
    with Image.open(path) as im:
        if source == "mask":
            im = im.convert("L")
            if im.size[0] != size:
                im = im.resize((size, size), Image.BOX)
            return np.asarray(im) > 127
        im = im.convert("RGB")
        if im.size[0] != size:
            im = im.resize((size, size), Image.BOX)
        return np.asarray(im, np.float32) / 255.0


def delta_e(a, b):
    """CIE76 colour difference per pixel, in the scorer's L*a*b*."""
    return np.sqrt(((rgb_to_lab(a) - rgb_to_lab(b)) ** 2).sum(-1))


def typical_tiles():
    """One scene per camera height (h00..h03), each the one whose shadow PSNR is closest to
    every scored model's median at once, among scenes with 30-60 % shadow (the middle of the
    split). "Typical", not "best": a reader should see what the table's mean looks like.

    Chosen among the scenes kept at full resolution when the models have any (`FULLRES`):
    the zoom and profile panels crop 384 px out of 2048, which a 512 px copy cannot give
    back. Without a retained set (a run scored before `eval.retain_predictions` that still
    has its images) the choice is over the whole split, exactly as before."""
    medians = {run: np.nanmedian(column(run, "psnr_shadow")) for run in AVAILABLE}
    spread = {run: np.nanstd(column(run, "psnr_shadow")) for run in AVAILABLE}
    fraction = dict(zip(KEYS, FRACTION))
    def score_of(key):
        return sum(abs(PER[r][key]["psnr_shadow"] - medians[r]) / spread[r] for r in AVAILABLE)

    chosen, used_locations = [], set()
    for height in ("h00", "h01", "h02", "h03"):
        pool = [key for key in KEYS if f"_{height}_" in key and 0.30 <= fraction[key] <= 0.60]
        # A full-resolution scene first (the zoom panels crop 384 px out of 2048), then one
        # whose location no other row uses. `eval.retain_predictions` keeps four test scenes,
        # so requiring both can leave a height with no candidate; each condition gives way
        # to the next instead of returning nothing for that row.
        best = None
        for tier in ([key for key in pool if key in FULLRES], pool):
            fresh = [key for key in tier if key.split("_")[0] not in used_locations]
            if fresh:
                best = min(fresh, key=score_of)
                break
        if best is None:  # every candidate shares its location with an earlier row
            best = min(pool, key=score_of)
        chosen.append(best)
        used_locations.add(best.split("_")[0])
    return chosen


TILES = typical_tiles()
for key in TILES:
    print(f"{key}  {altitude_m(key):5.0f} m  shadow {PER[AVAILABLE[0]][key]['shadow_fraction']:.0%}  " +
          "  ".join(f"{LABEL[r]} {PER[r][key]['psnr_shadow']:.2f} dB" for r in AVAILABLE))


## Figure 1 -- the qualitative grid

One typical scene per camera height (rows, low to high), with the share of the scene the
shadow mask covers. The number in each prediction is that image's PSNR inside the shadow,
so the panel and the table can be read against each other. (The mask itself is drawn in
figures 3 and 4: over tree canopy its outline is a speckle that hides the picture.)

In [ ]:
def badge(ax, text, loc="lower left"):
    x, ha = (0.03, "left") if "left" in loc else (0.97, "right")
    y, va = (0.04, "bottom") if "lower" in loc else (0.96, "top")
    ax.text(x, y, text, transform=ax.transAxes, ha=ha, va=va, fontsize=7.5, color=INK,
            bbox=dict(boxstyle="round,pad=0.25", fc=(1, 1, 1, 0.82), ec="none"))


def outline(ax, key, size, coarse=128, **kw):
    """The shadow mask's outline at `coarse` px, drawn on a `size` px pane. Under tree canopy
    the full-resolution mask is a speckle of leaves, and its outline paints the panel orange;
    at 128 px it keeps the shapes a reader looks for (building and tree shadows)."""
    m = load("mask", key, coarse).astype(float)
    ax.contour(m, levels=[0.5], colors=[MASK_EDGE], linewidths=kw.get("lw", 0.9),
               extent=(-0.5, size - 0.5, size - 0.5, -0.5), origin="upper")


def fig_grid(tiles=TILES, size=512):
    cols = ["input", *AVAILABLE, "target"]
    titles = ["Input", *[LABEL[r] for r in AVAILABLE], "Target"]
    width = 7.2
    cell = width / len(cols)
    fig, axes = plt.subplots(len(tiles), len(cols), figsize=(width, cell * len(tiles) + 0.25),
                             squeeze=False)
    for i, key in enumerate(tiles):
        mask = load("mask", key, size)
        for j, source in enumerate(cols):
            ax = axes[i, j]
            strip(ax, load(source, key, size))
            if source == "input":
                ax.set_ylabel(f"{altitude_m(key):.0f} m", color=INK2, fontsize=9)
                badge(ax, f"{mask.mean():.0%} shadow")
            elif source != "target":
                badge(ax, f"{PER[source][key]['psnr_shadow']:.1f} dB")
            if i == 0:
                # Eight columns in a 7.2 in figure leave 0.9 in each: "ShadowMaskFormer" at
                # 9 pt is wider than its cell and the headers run into one another.
                ax.set_title(titles[j], fontsize=7.2, loc="center")
    fig.subplots_adjust(left=0.03, right=0.999, top=1 - 0.25 / fig.get_figheight(), bottom=0.002,
                        hspace=0.025, wspace=0.025)
    save(fig, "shadow_fig1_grid")


fig_grid()

## Figure 2 -- before / after, split diagonally

The same scene on both sides of the diagonal: shadowed input above, the model's output
below. The seam is where the eye compares, so a shadow that was only darkened less (not
removed) shows as a step across it. The last column splits input and ground truth -- what
a perfect removal would look like. Made for the project page as much as for the paper.

In [ ]:
def split_pane(ax, upper, lower, label_upper="input", label_lower="output"):
    n = upper.shape[0]
    yy, xx = np.mgrid[0:n, 0:n]
    canvas = np.where((xx + yy < n)[..., None], upper, lower)
    strip(ax, canvas)
    ax.plot([0, n - 1], [n - 1, 0], color="white", lw=1.4, solid_capstyle="butt")
    badge(ax, label_upper, "upper left")
    badge(ax, label_lower, "lower right")


def fig_split(tiles=TILES[:2], size=768):
    cols = [*AVAILABLE, "target"]
    width = 7.2
    cell = width / len(cols)
    fig, axes = plt.subplots(len(tiles), len(cols), figsize=(width, cell * len(tiles) + 0.25),
                             squeeze=False)
    for i, key in enumerate(tiles):
        before = load("input", key, size)
        for j, source in enumerate(cols):
            name = "ground truth" if source == "target" else LABEL[source]
            split_pane(axes[i, j], before, load(source, key, size), "input", name)
            if i == 0:
                axes[i, j].set_title("Target" if source == "target" else LABEL[source],
                                     fontsize=9.5, loc="center")
    fig.subplots_adjust(left=0.002, right=0.998, top=1 - 0.25 / fig.get_figheight(), bottom=0.002,
                        hspace=0.02, wspace=0.02)
    save(fig, "shadow_fig2_split")


fig_split()

## Figure 3 -- where the error sits (zoom + colour-difference maps)

A 384 px window at full resolution, placed automatically where the shadow outline is
densest (so it contains both sides of several shadow edges). Top row: the pixels. Bottom
row: CIE76 colour difference to the target, one scale for every panel; the first map is
the input itself, i.e. what the shadow costs before any removal. The amber line is the
shadow mask.

In [ ]:
CROP = 384


def busiest_window(mask, size=CROP, stride=64):
    """Top-left corner of the window with the most shadow-edge pixels (and some of both sides)."""
    edge = (mask ^ np.roll(mask, 1, 0)) | (mask ^ np.roll(mask, 1, 1))
    best, where = -1, (0, 0)
    for y in range(0, mask.shape[0] - size + 1, stride):
        for x in range(0, mask.shape[1] - size + 1, stride):
            window = mask[y:y + size, x:x + size]
            if not 0.25 <= window.mean() <= 0.75:
                continue
            count = edge[y:y + size, x:x + size].sum()
            if count > best:
                best, where = count, (y, x)
    return where


def fig_zoom(key=TILES[0]):
    mask = load("mask", key)
    y0, x0 = busiest_window(mask)
    win = (slice(y0, y0 + CROP), slice(x0, x0 + CROP))
    target = load("target", key)[win]
    m = mask[win]
    sources = ["input", *AVAILABLE]
    errors = {s: delta_e(load(s, key)[win], target) for s in sources}
    vmax = float(np.percentile(errors["input"], 99))

    n = len(sources) + 1  # + the target column
    fig = plt.figure(figsize=(7.2, 7.2 * 2 / (n + 2) + 0.35))
    gs = GridSpec(2, n + 2, figure=fig, wspace=0.03, hspace=0.04,
                  left=0.002, right=0.94, top=1 - 0.3 / fig.get_figheight(), bottom=0.004)
    overview = fig.add_subplot(gs[:, :2])
    strip(overview, load("input", key, 1024))
    s = 1024 / 2048
    overview.add_patch(Rectangle((x0 * s, y0 * s), CROP * s, CROP * s, fill=False,
                                 edgecolor="white", lw=1.6))
    overview.add_patch(Rectangle((x0 * s, y0 * s), CROP * s, CROP * s, fill=False,
                                 edgecolor=INK, lw=0.8, linestyle=(0, (3, 2))))
    overview.set_title(f"Input, {altitude_m(key):.0f} m", fontsize=9, loc="left")

    columns = ["input", "target", *AVAILABLE]
    for j, source in enumerate(columns):
        top = fig.add_subplot(gs[0, j + 2])
        strip(top, target if source == "target" else load(source, key)[win])
        top.contour(m, levels=[0.5], colors=[MASK_EDGE], linewidths=0.6)
        top.set_title({"input": "Input", "target": "Target"}.get(source, LABEL.get(source)),
                      fontsize=8.5, loc="center")
        bottom = fig.add_subplot(gs[1, j + 2])
        if source == "target":
            strip(bottom, np.where(m[..., None], 0.18, 0.96) * np.ones(3))
            badge(bottom, "shadow mask", "lower left")
            continue
        strip(bottom, errors[source], cmap=SEQ_ERR, vmin=0, vmax=vmax)
        im = bottom.images[-1]
        bottom.contour(m, levels=[0.5], colors=[MASK_EDGE], linewidths=0.5)
        e = errors[source]
        badge(bottom, f"shadow {e[m].mean():.1f}\nlit {e[~m].mean():.1f}", "lower left")
    cax = fig.add_axes((0.948, 0.06, 0.012, 0.36))
    bar = fig.colorbar(im, cax=cax)
    bar.outline.set_visible(False)
    bar.ax.tick_params(labelsize=7.5, length=2)
    bar.set_label(r"$\Delta E^*_{ab}$ to target", fontsize=8, color=INK2)
    save(fig, "shadow_fig3_zoom_error")


fig_zoom()

## Figure 4 -- a lightness profile across shadow edges

One line through the same window, chosen to cross the most shadow edges. The grey bands
are the shadowed stretches. A removal that works lifts the input (grey) onto the target
(black) inside the bands and leaves it alone outside them; a step at a band edge is a
shadow boundary the model left behind.

In [ ]:
def fig_profile(key=TILES[0], band=5, smooth=7):
    mask = load("mask", key)
    y0, x0 = busiest_window(mask)
    win = (slice(y0, y0 + CROP), slice(x0, x0 + CROP))
    m = mask[win]
    row = int(np.argmax([(np.diff(m[r].astype(int)) != 0).sum() for r in range(band, CROP - band)])) + band
    rows = slice(row - band // 2, row + band // 2 + 1)
    kernel = np.ones(smooth) / smooth

    def lightness(source):
        image = load("target", key)[win] if source == "target" else load(source, key)[win]
        return np.convolve(rgb_to_lab(image[rows])[..., 0].mean(0), kernel, mode="same")

    fig = plt.figure(figsize=(7.2, 2.35))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[1, 3.2], wspace=0.04,
                  left=0.0, right=0.995, top=0.9, bottom=0.17)
    pane = fig.add_subplot(gs[0])
    strip(pane, load("input", key)[win])
    pane.contour(m, levels=[0.5], colors=[MASK_EDGE], linewidths=0.6)
    pane.axhline(row, color="white", lw=2.2)
    pane.axhline(row, color=INK, lw=0.9, linestyle=(0, (3, 2)))

    ax = fig.add_subplot(gs[1])
    shadowed = m[row]
    starts = np.flatnonzero(np.diff(np.r_[0, shadowed.astype(int)]) == 1)
    ends = np.flatnonzero(np.diff(np.r_[shadowed.astype(int), 0]) == -1)
    for a, b in zip(starts, ends):
        ax.axvspan(a - 0.5, b + 0.5, color=HAIR, lw=0, zorder=0)
    xs = np.arange(CROP)
    ax.plot(xs, lightness("input"), color=INPUT_C, lw=1.4, linestyle=(0, (4, 2)), label="Input")
    for run in AVAILABLE:
        ax.plot(xs, lightness(run), color=COLOR[run], lw=1.5, label=LABEL[run])
    ax.plot(xs, lightness("target"), color=TARGET_C, lw=1.7, label="Target")
    tidy(ax)
    ax.set_xlim(0, CROP - 1)
    ax.set_xlabel("pixel along the line", fontsize=9)
    ax.set_ylabel(r"$L^*$", fontsize=9)
    ax.yaxis.set_label_position("right"), ax.yaxis.tick_right()
    ax.legend(ncol=len(AVAILABLE) + 2, loc="lower center", bbox_to_anchor=(0.5, 1.0),
              fontsize=8, handlelength=1.6, columnspacing=1.0, borderaxespad=0.1)
    save(fig, "shadow_fig4_profile")


fig_profile()

## Figure 5 -- is the shadow lifted to the right brightness?

The distribution of $L^*$ over 150 test scenes (every 20th, reduced to 512 px), inside
the shadow mask (left) and outside it (right). The target's curve is where a removal
should land. A model whose shadow curve stops short of the target only brightened the
shadow; one that also shifts the lit curve is changing pixels it should not touch.

In [ ]:
def lightness_samples(step=20, size=512, cap=40_000):
    path = CACHE / f"lightness_{step}_{size}_{'_'.join(AVAILABLE)}.npz"
    if path.is_file():
        return dict(np.load(path))
    rng = np.random.default_rng(0)
    out = {f"{s}_{region}": [] for s in ("input", "target", *AVAILABLE) for region in ("shadow", "lit")}
    for key in KEYS[::step]:
        m = load("mask", key, size)
        for s in ("input", "target", *AVAILABLE):
            lightness = rgb_to_lab(load(s, key, size))[..., 0]
            for region, sel in (("shadow", m), ("lit", ~m)):
                values = lightness[sel]
                if values.size > cap // 10:
                    values = rng.choice(values, cap // 10, replace=False)
                out[f"{s}_{region}"].append(values)
    out = {k: np.concatenate(v) for k, v in out.items()}
    np.savez_compressed(path, **out)
    return out


def density(values, edges, smooth=3):
    hist, _ = np.histogram(values, edges, density=True)
    return np.convolve(hist, np.ones(smooth) / smooth, mode="same")


def fig_lightness():
    samples = lightness_samples()
    edges = np.linspace(0, 100, 101)
    centres = (edges[:-1] + edges[1:]) / 2
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.5), sharey=True)
    for ax, region, title in zip(axes, ("shadow", "lit"), ("Inside the shadow mask", "Outside it (lit)")):
        ax.fill_between(centres, density(samples[f"target_{region}"], edges), color=HAIR, lw=0)
        ax.plot(centres, density(samples[f"target_{region}"], edges), color=TARGET_C, lw=1.7, label="Target")
        ax.plot(centres, density(samples[f"input_{region}"], edges), color=INPUT_C, lw=1.4,
                linestyle=(0, (4, 2)), label="Input")
        for run in AVAILABLE:
            ax.plot(centres, density(samples[f"{run}_{region}"], edges), color=COLOR[run], lw=1.5,
                    label=LABEL[run])
        for s, c in (("input", INPUT_C), ("target", TARGET_C), *((r, COLOR[r]) for r in AVAILABLE)):
            ax.axvline(np.median(samples[f"{s}_{region}"]), color=c, lw=0.8, ymax=0.06)
        tidy(ax)
        ax.set_title(title, fontsize=9.5)
        ax.set_xlim(0, 100)
        ax.set_xticks([0, 25, 50, 75, 100], ["0", "25", "50", "75", ""])
        ax.set_xlabel(r"$L^*$  (ticks on the axis: medians)", fontsize=9)
        ax.set_yticks([])
    axes[0].set_ylabel("density", fontsize=9)
    axes[1].legend(loc="upper right", fontsize=8, handlelength=1.6)
    fig.subplots_adjust(left=0.035, right=0.998, top=0.9, bottom=0.18, wspace=0.06)
    save(fig, "shadow_fig5_lightness")
    print("median L*:", {s: {r: round(float(np.median(samples[f'{s}_{r}'])), 1) for r in ("shadow", "lit")}
                         for s in ("input", "target", *AVAILABLE)})


fig_lightness()

## Figure 6 -- shadow against lit pixels, per metric (dumbbells)

For every model, the mean over the 3 000 test scenes inside the shadow (filled) and
outside it (open), joined by a line whose length is the gap the shadow still costs; the
tick is the whole image. The "Input" row is the shadowed image scored as if it were the
answer -- the floor every model has to clear (PSNR on 600 scenes, every 5th; computed here
with the scorer's own PSNR definition, not re-scored for the models).

In [ ]:
def identity_psnr(step=5):
    """PSNR of the untouched input against the target, all / shadow / lit, per scene."""
    path = CACHE / f"identity_psnr_{step}.json"
    if path.is_file():
        return json.loads(path.read_text())
    rows = []
    for key in KEYS[::step]:
        a, b, m = load("input", key), load("target", key), load("mask", key)
        squared = ((a - b) ** 2).mean(-1)
        row = {}
        for name, sel in (("psnr", None), ("psnr_shadow", m), ("psnr_nonshadow", ~m)):
            mse = squared.mean() if sel is None else squared[sel].mean()
            row[name] = float(10 * np.log10(1.0 / max(float(mse), 1e-12)))
        rows.append(row)
        load.cache_clear()  # 2048 px images: keep the cache for the figures, not this loop
    path.write_text(json.dumps(rows))
    return rows


IDENTITY = identity_psnr()


def fig_regions():
    panels = [("psnr", "PSNR (dB)", True), ("ssim_y", "SSIM (Y)", True), ("mae_lab", r"MAE $L^*a^*b^*$", False)]
    rows = ["input", *AVAILABLE]
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 0.42 * len(rows) + 0.75), sharey=True)
    for ax, (metric, title, higher) in zip(axes, panels):
        for i, run in enumerate(rows):
            y = len(rows) - 1 - i
            if run == "input":
                if metric != "psnr":
                    continue
                values = [np.mean([r[k] for r in IDENTITY]) for k in ("psnr", "psnr_shadow", "psnr_nonshadow")]
                color = INPUT_C
            else:
                values = [np.nanmean(column(run, metric + suffix)) for suffix in ("", "_shadow", "_nonshadow")]
                color = COLOR[run]
            whole, shadow, lit = values
            ax.plot([shadow, lit], [y, y], color=color, lw=2.2, solid_capstyle="round", zorder=2)
            ax.scatter([shadow], [y], s=46, color=color, zorder=3, edgecolor="white", linewidth=1.2)
            ax.scatter([lit], [y], s=46, facecolor="white", edgecolor=color, linewidth=1.8, zorder=3)
            ax.plot([whole, whole], [y - 0.22, y + 0.22], color=INK, lw=1.2, zorder=4)
        tidy(ax, "x")
        ax.set_title(title + ("  ↑" if higher else "  ↓"), fontsize=9.5)
        ax.margins(x=0.12)
        ax.set_ylim(-0.6, len(rows) - 0.4)
    axes[0].set_yticks(range(len(rows)))
    axes[0].set_yticklabels([("Input as-is" if r == "input" else LABEL[r]) for r in rows][::-1])
    handles = [Line2D([], [], marker="o", ls="", color=INK2, markeredgecolor="white", markersize=7, label="shadow"),
               Line2D([], [], marker="o", ls="", markerfacecolor="white", markeredgecolor=INK2, markersize=7, label="lit"),
               Line2D([], [], marker="|", ls="", color=INK, markersize=9, label="whole image")]
    fig.legend(handles=handles, loc="lower center", ncol=3, bbox_to_anchor=(0.6, -0.02), fontsize=8.5)
    fig.subplots_adjust(left=0.2, right=0.995, top=0.88, bottom=0.27, wspace=0.08)
    save(fig, "shadow_fig6_regions")


fig_regions()

## Figure 7 -- every scene: shadow PSNR against lit PSNR

One hexbin per model over the 3 000 test scenes. Below the diagonal the shadowed pixels
are reconstructed worse than the lit ones in the same image; the share of scenes there is
printed in each panel. (The untouched input is off this scale -- 27 dB lit, 7 dB in the
shadow on average -- and is shown in figures 6 and 8 instead.)

In [ ]:
def fig_scatter():
    n = len(AVAILABLE)
    cols = min(n, 3)
    nrows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(nrows, cols, figsize=(7.2, 7.2 / cols * nrows * 0.92 + 0.1),
                             sharex=True, sharey=True, squeeze=False)
    lo = np.nanpercentile(np.concatenate([column(r, "psnr_shadow") for r in AVAILABLE]), 0.5)
    hi = np.nanpercentile(np.concatenate([column(r, "psnr_nonshadow") for r in AVAILABLE]), 99.5)
    for ax, run in zip(axes.ravel(), AVAILABLE):
        x, y = column(run, "psnr_nonshadow"), column(run, "psnr_shadow")
        ok = np.isfinite(x) & np.isfinite(y)
        ax.hexbin(x[ok], y[ok], gridsize=34, extent=(lo, hi, lo, hi), cmap=SEQ, mincnt=1,
                  linewidths=0.2, edgecolors="white")
        ax.plot([lo, hi], [lo, hi], color=INK, lw=0.8)
        tidy(ax, "both")
        ax.set_xlim(lo, hi), ax.set_ylim(lo, hi)
        ax.set_aspect("equal")
        ax.set_title(LABEL[run], fontsize=9.5)
        badge(ax, f"{np.mean(y[ok] < x[ok]):.0%} of scenes\nbelow the diagonal", "upper left")
    for ax in axes.ravel()[n:]:
        ax.set_visible(False)
    for ax in axes[-1]:
        ax.set_xlabel("PSNR, lit pixels (dB)", fontsize=9)
    for ax in axes[:, 0]:
        ax.set_ylabel("PSNR, shadow (dB)", fontsize=9)
    fig.subplots_adjust(left=0.09, right=0.995, top=0.94, bottom=0.1, wspace=0.08, hspace=0.16)
    save(fig, "shadow_fig7_perscene")


fig_scatter()

## Figure 8 -- does it depend on how much of the scene is in shadow?

The scenes binned into eighths by shadow fraction; line = mean PSNR inside the shadow
(left) and outside it (right), band = interquartile range, dashed grey = the untouched
input. If the lit curve falls as the shadow grows, the model is spending its errors on
the lit pixels of heavily shadowed scenes.

In [ ]:
def fig_fraction(nbins=8):
    edges = np.quantile(FRACTION, np.linspace(0, 1, nbins + 1))
    centres = 0.5 * (edges[:-1] + edges[1:])
    which = np.clip(np.digitize(FRACTION, edges[1:-1]), 0, nbins - 1)
    identity_fraction = np.array([PER[AVAILABLE[0]][k]["shadow_fraction"] for k in KEYS[::5]])
    identity_bin = np.clip(np.digitize(identity_fraction, edges[1:-1]), 0, nbins - 1)
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.6), sharey=True)
    for ax, metric, title in zip(axes, ("psnr_shadow", "psnr_nonshadow"), ("Shadow pixels", "Lit pixels")):
        base = np.array([r[metric] for r in IDENTITY])
        ax.plot(centres * 100, [base[identity_bin == b].mean() for b in range(nbins)], color=INPUT_C,
                lw=1.4, linestyle=(0, (4, 2)), label="Input")
        for run in AVAILABLE:
            v = column(run, metric)
            mean = [np.nanmean(v[which == b]) for b in range(nbins)]
            q1 = [np.nanpercentile(v[which == b], 25) for b in range(nbins)]
            q3 = [np.nanpercentile(v[which == b], 75) for b in range(nbins)]
            ax.fill_between(centres * 100, q1, q3, color=COLOR[run], alpha=0.14, lw=0)
            ax.plot(centres * 100, mean, color=COLOR[run], marker="o", markersize=4.5,
                    markeredgecolor="white", markeredgewidth=0.8, label=LABEL[run])
        tidy(ax)
        ax.set_title(title, fontsize=9.5)
        ax.set_xlabel("shadow fraction of the scene (%)", fontsize=9)
    axes[0].set_ylabel("PSNR (dB)", fontsize=9)
    axes[1].legend(loc="lower left", fontsize=8, handlelength=1.6)
    fig.subplots_adjust(left=0.075, right=0.998, top=0.9, bottom=0.18, wspace=0.05)
    save(fig, "shadow_fig8_fraction")


fig_fraction()

In [ ]:
# Numbers the text quotes, printed once so a caption never has to guess.
print("--- summary for the text ---")
base = {k: np.mean([r[k] for r in IDENTITY]) for k in ("psnr", "psnr_shadow", "psnr_nonshadow")}
print(f"input as-is ({len(IDENTITY)} scenes): PSNR {base['psnr']:.2f} dB, shadow {base['psnr_shadow']:.2f}, "
      f"lit {base['psnr_nonshadow']:.2f}")
for run in AVAILABLE:
    s, l = np.nanmean(column(run, "psnr_shadow")), np.nanmean(column(run, "psnr_nonshadow"))
    print(f"{LABEL[run]:<17} shadow {s:.2f} dB ({s - base['psnr_shadow']:+.2f} vs input), "
          f"lit {l:.2f} dB ({l - base['psnr_nonshadow']:+.2f} vs input), gap lit-shadow {l - s:.2f} dB")